# Project Track 1 - Reynolds-Number Generalization

**Choose one variant:**

- **1A Interpolation:** quantify performance at unseen Reynolds numbers inside the training range.
- **1B Extrapolation:** train only on lower-Re cases and determine where the surrogate fails at higher Re.

The notebook supplies the solver output, interpolation baseline, coordinate DNN, metrics, and plotting functions. Your work is to freeze a split, select an architecture using validation only, perform a controlled comparison, and defend a conclusion.

## Required files
`P1_Re_Generalization.ipynb`, `w4utils.py`, `w5_common.py`, `cavity_data.npz`

In [ ]:
import time, importlib, ast
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import w4utils, w5_common
importlib.reload(w4utils); importlib.reload(w5_common)
w5_common.set_global_seed(690)
assert w4utils.W4_UTILS_VERSION == "6.3", "Upload the revised w4utils.py (v6.3)."
assert w5_common.W5_COMMON_VERSION == "1.2", "Upload the revised w5_common.py (v1.2)."
data=w5_common.require_week4_files()

## 1. Freeze the scientific split before training

The two variants intentionally answer different questions. Do not move a test case into training after viewing its result.

In [ ]:
VARIANT="1A"  # EDIT to "1B" only if Track 1B was approved.

if VARIANT=="1A":
    TRAIN_RE=[100,150,200,250,300,400]
    VAL_RE=225
    TEST_RE=[175,275,350,375]
    CLAIM="interpolation inside the covered Reynolds-number range"
elif VARIANT=="1B":
    TRAIN_RE=[100,150,200,225,250]
    VAL_RE=300
    TEST_RE=[350,375,400]
    CLAIM="higher-Re extrapolation beyond the training range"
else:
    raise ValueError("VARIANT must be 1A or 1B")

print(CLAIM); print("train",TRAIN_RE,"validation",VAL_RE,"blind test",TEST_RE)

## 2. Architecture selection is a validation experiment

All candidates use the same data, optimizer, and stopping rule. Select by Re=`VAL_RE`; never select by blind-test error.

In [ ]:
CANDIDATES=[(32,32),(64,64,64),(96,96)]
selection=[]; bundles={}
for hidden in CANDIDATES:
    t0=time.time()
    b=w5_common.train_pointwise_model(data,TRAIN_RE,VAL_RE,hidden=hidden,
        stride=2,seed=690,epochs=850,patience=60)
    pred=w5_common.predict_case(b,VAL_RE,data["x"],data["y"])
    rep=w5_common.evaluate_prediction(data,VAL_RE,pred)
    selection.append({"hidden":str(hidden),"best_epoch":b["best_epoch"],
                      "val_relative_L2_uv":rep["relative_L2_uv"],
                      "val_relative_L2_p":rep["relative_L2_p"],
                      "seconds":time.time()-t0})
    bundles[hidden]=b
selection=pd.DataFrame(selection).sort_values("val_relative_L2_uv")
display(selection)
BEST=ast.literal_eval(selection.iloc[0]["hidden"])
FINAL_EPOCHS=int(selection.iloc[0]["best_epoch"])
print("Frozen architecture:",BEST)
print("Frozen full-development epoch budget:",FINAL_EPOCHS)

## 3. Retrain once on **all** permitted development cases

After architecture and epoch budget are frozen using `VAL_RE`, add the validation case to the development set and retrain once with a fixed number of epochs. No development Reynolds number is held out during this final fit. This gives the DNN and the field-interpolation baseline exactly the same Reynolds-number information. Blind-test cases remain unopened.

In [ ]:
DEV_RE=sorted(TRAIN_RE+[VAL_RE])
final_bundle=w5_common.train_pointwise_fixed_epochs(
    data,DEV_RE,hidden=BEST,stride=2,seed=690,
    epochs=FINAL_EPOCHS,learning_rate=1e-3)
print("Final DNN trained on all development cases:",DEV_RE)

## 4. Open the blind cases and compare fairly against field interpolation

Both methods below use the same `DEV_RE` cases. The required comparison is **not** just a global RMSE table. Include centerline, wall, divergence, and pressure-interior evidence.

**Important expectation:** for this smooth, fixed-grid cavity family, direct field interpolation is a very strong baseline and may outperform the DNN by a wide margin. A well-supported negative result is scientifically valid; do not tune the blind cases simply to make the neural model win.

In [ ]:
rows=[]; stored={}
for r in TEST_RE:
    interp=w5_common.interpolate_case(data,r,DEV_RE)
    dnn=w5_common.predict_case(final_bundle,r,data["x"],data["y"])
    stored[r]={"interpolation":interp,"coordinate DNN":dnn}
    for method,pred in stored[r].items():
        rows.append({"variant":VARIANT,"method":method,"Re":r,
                     **w5_common.evaluate_prediction(data,r,pred)})
results=w5_common.results_frame(rows)
results.to_csv("P1_results.csv",index=False)
display(results)

In [ ]:
# Select the most informative blind case, not merely the prettiest result.
CASE_TO_PLOT=TEST_RE[-1] if VARIANT=="1B" else 275
fig=w5_common.plot_case_evidence(data,CASE_TO_PLOT,stored[CASE_TO_PLOT],
                                 f"Track {VARIANT}: blind evidence")
plt.show()

## Required student decisions and report evidence

1. Explain why your split measures interpolation or extrapolation.
2. Show the validation-only architecture table.
3. Report both interpolation and DNN metrics for every blind case.
4. Include two velocity centerlines and at least one pressure profile.
5. Use wall error and divergence as physical checks.
6. Identify the first Reynolds number at which the model becomes unreliable under a criterion you state in advance.
7. Explain whether the neural model justifies its complexity over interpolation.

**Do not:** change the split after test results, tune on Re=375/400, or claim generalization from one contour plot.

**Fairness check:** confirm in the report that both the DNN and interpolation baseline used exactly the same `DEV_RE` list.

## Optional stretch extensions (advanced / prize-track only)

Complete the required project first. With instructor approval, choose at most one:

1. Train a small ensemble and test whether ensemble spread rises near the observed Reynolds-number failure boundary.
2. Compare a validation-derived rejection rule against actual blind error and discuss false confidence.
3. Compare interpolation, the coordinate DNN, and one reduced-order coefficient model under the same development cases and storage/runtime accounting.
